In [1]:
import numpy as np
import pandas as pd
from math import log2
from scipy.optimize import minimize

pd.set_option("display.width", 120)

data = pd.DataFrame({
    "Wind":     ["Weak", "Strong", "Weak", "Weak", "Strong"],
    "Humidity": ["High", "High",   "Low",  "Low",  "Low"],
    "Rain":     ["Yes",  "Yes",    "No",   "Yes",  "No"],
})
data

,Wind,Humidity,Rain
0,Weak,High,Yes
1,Strong,High,Yes
2,Weak,Low,No
3,Weak,Low,Yes
4,Strong,Low,No


## Part 1

In [ ]:
classes = ["Yes", "No"]
prior = data["Rain"].value_counts(normalize=True)
print("Priors:  P(Yes) =", round(prior["Yes"], 2), "  P(No) =", round(prior["No"], 2))

def laplace_likelihood(attribute, value, cls, k=2):
    in_class = data[data["Rain"] == cls]
    count = (in_class[attribute] == value).sum()
    return (count + 1) / (len(in_class) + k)

likelihoods = pd.DataFrame(
    {c: [laplace_likelihood("Wind", "Weak", c), laplace_likelihood("Wind", "Strong", c),
         laplace_likelihood("Humidity", "High", c), laplace_likelihood("Humidity", "Low", c)]
     for c in classes},
    index=["P(Weak | class)", "P(Strong | class)", "P(High | class)", "P(Low | class)"])
likelihoods.round(3)

Priors:  P(Yes) = 0.6   P(No) = 0.4


,Yes,No
P(Weak | class),0.6,0.50
P(Strong | class),0.4,0.50
P(High | class),0.6,0.25
P(Low | class),0.4,0.75


In [ ]:
yes_score = prior["Yes"] * laplace_likelihood("Wind", "Weak", "Yes") * laplace_likelihood("Humidity", "High", "Yes")
no_score  = prior["No"]  * laplace_likelihood("Wind", "Weak", "No")  * laplace_likelihood("Humidity", "High", "No")
print(f"Yes score = {prior['Yes']:.1f} x {laplace_likelihood('Wind','Weak','Yes'):.1f} x {laplace_likelihood('Humidity','High','Yes'):.1f} = {yes_score:.3f}")
print(f"No score  = {prior['No']:.1f} x {laplace_likelihood('Wind','Weak','No'):.1f} x {laplace_likelihood('Humidity','High','No'):.2f} = {no_score:.3f}")
print(f"Normalized P(Yes) = {yes_score/(yes_score+no_score):.1%}")

Yes score = 0.6 x 0.6 x 0.6 = 0.216
No score  = 0.4 x 0.5 x 0.25 = 0.050
Normalized P(Yes) = 81.2%


In [4]:
rows = []
for wind in ["Weak", "Strong"]:
    for hum in ["High", "Low"]:
        score = {c: prior[c] * laplace_likelihood("Wind", wind, c) * laplace_likelihood("Humidity", hum, c)
                 for c in classes}
        total = score["Yes"] + score["No"]
        rows.append({
            "Wind": wind, "Humidity": hum,
            "Score Yes": round(score["Yes"], 3), "Score No": round(score["No"], 3),
            "P(Yes)": f"{score['Yes'] / total:.1%}", "P(No)": f"{score['No'] / total:.1%}",
            "Decision": "Hold (rain)" if score["Yes"] > score["No"] else "Deliver",
        })
nb_results = pd.DataFrame(rows)
nb_results

,Wind,Humidity,Score Yes,Score No,P(Yes),P(No),Decision
0,Weak,High,0.216,0.05,81.2%,18.8%,Hold (rain)
1,Weak,Low,0.144,0.15,49.0%,51.0%,Deliver
2,Strong,High,0.144,0.05,74.2%,25.8%,Hold (rain)
3,Strong,Low,0.096,0.15,39.0%,61.0%,Deliver


In [ ]:
raw_p_high_given_no = ((data["Rain"] == "No") & (data["Humidity"] == "High")).sum() / (data["Rain"] == "No").sum()
print("Unsmoothed P(High | No) =", raw_p_high_given_no)
print("Without smoothing, Weak + High would get P(No) = 0, so the drone would always hold the parcel when humidity is High.")

Unsmoothed P(High | No) = 0.0
Without smoothing, Weak + High would get P(No) = 0, so the drone would always hold the parcel when humidity is High.


## Part 2: 

In [6]:
def entropy(counts):
    total = sum(counts)
    return -sum((c / total) * log2(c / total) for c in counts if c > 0)

def information_gain(df, attribute, target="Rain"):
    parent = entropy(df[target].value_counts().tolist())
    weighted_child = 0.0
    for _, subset in df.groupby(attribute):
        weighted_child += len(subset) / len(df) * entropy(subset[target].value_counts().tolist())
    return parent - weighted_child

print(f"H(S) = {entropy(data['Rain'].value_counts().tolist()):.3f}")
for attribute in ["Wind", "Humidity"]:
    print(f"IG({attribute}) = {information_gain(data, attribute):.3f}")
print("\nHumidity has the higher gain, so it becomes the root.")

H(S) = 0.971
IG(Wind) = 0.020
IG(Humidity) = 0.420

Humidity has the higher gain, so it becomes the root.


In [7]:
low = data[data["Humidity"] == "Low"]
print(f"Humidity = Low branch: H = {entropy(low['Rain'].value_counts().tolist()):.3f}")
print(f"IG(Wind | Humidity = Low) = {information_gain(low, 'Wind'):.3f}")
print()
print("Tree:")
print("  Humidity = High -> Rain = Yes (pure leaf)")
print("  Humidity = Low  -> split on Wind:")
print("      Strong -> Rain = No (pure leaf)")
print("      Weak   -> 1 Yes / 1 No (rows 3 and 4 have identical inputs), tie broken by the Low-branch majority (2 No vs 1 Yes) -> No")

Humidity = Low branch: H = 0.918
IG(Wind | Humidity = Low) = 0.252

Tree:
  Humidity = High -> Rain = Yes (pure leaf)
  Humidity = Low  -> split on Wind:
      Strong -> Rain = No (pure leaf)
      Weak   -> 1 Yes / 1 No (rows 3 and 4 have identical inputs), tie broken by the Low-branch majority (2 No vs 1 Yes) -> No


In [ ]:
def tree_predict(wind, hum):
    if hum == "High":
        return "Yes"
    return "No"

tree_results = pd.DataFrame([
    {"Wind": w, "Humidity": h, "Tree prediction": tree_predict(w, h),
     "Decision": "Hold (rain)" if tree_predict(w, h) == "Yes" else "Deliver"}
    for w in ["Weak", "Strong"] for h in ["High", "Low"]
])
tree_results

,Wind,Humidity,Tree prediction,Decision
0,Weak,High,Yes,Hold (rain)
1,Weak,Low,No,Deliver
2,Strong,High,Yes,Hold (rain)
3,Strong,Low,No,Deliver


## Part 3: 

In [ ]:
def encode(wind, hum):
    return np.array([0 if wind == "Weak" else 1, 0 if hum == "Low" else 1])

X = np.array([encode(w, h) for w, h in zip(data["Wind"], data["Humidity"])])
y = np.array([1 if r == "Yes" else 0 for r in data["Rain"]])
print("X (Wind, Humidity) =")
print(X)
print("y (Rain: Yes = 1, No = 0) =", y)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def lr_prob(wind, hum, w):
    x = encode(wind, hum)
    return sigmoid(w[0] + w[1] * x[0] + w[2] * x[1])

def gradient(w):
    p = sigmoid(w[0] + X @ w[1:])
    r = p - y
    return np.array([r.mean(), (r * X[:, 0]).mean(), (r * X[:, 1]).mean()])

lr = 1.0  # learning rate

X (Wind, Humidity) =
[[0 1]
 [1 1]
 [0 0]
 [0 0]
 [1 0]]
y (Rain: Yes = 1, No = 0) = [1 1 0 1 0]


In [ ]:
w = np.zeros(3)   
for it in (1, 2):
    g = gradient(w)
    w = w - lr * g
    print(f"Iteration {it}: gradient = {np.round(g, 4)}   new w = {np.round(w, 4)}")

Iteration 1: gradient = [-0.1  0.  -0.2]   new w = [0.1 0.  0.2]
Iteration 2: gradient = [-0.0552  0.0199 -0.1702]   new w = [ 0.1552 -0.0199  0.3702]


In [ ]:
w = np.zeros(3)
snapshots = {}
for it in range(1, 1001):
    w = w - lr * gradient(w)
    if it in (2, 10, 100, 1000):
        snapshots[it] = w.copy()

pd.DataFrame(snapshots, index=["bias", "w_wind", "w_humidity"]).round(4)

,2,10,100,1000
bias,0.1552,0.1338,-0.0741,-0.0098
w_wind,-0.0199,-0.3411,-1.9128,-4.1956
w_humidity,0.3702,1.2951,4.5159,8.8324


In [12]:
w_final = snapshots[1000]
cases = [("Weak", "High"), ("Strong", "High"), ("Weak", "Low"), ("Strong", "Low")]
lr_results = []
for wind, hum in cases:
    p = lr_prob(wind, hum, w_final)
    lr_results.append({"Wind": wind, "Humidity": hum, "P(Yes)": f"{p:.2%}", "P(No)": f"{1 - p:.2%}",
                       "Decision": "Hold (rain)" if p >= 0.5 else "Deliver"})
pd.DataFrame(lr_results)

,Wind,Humidity,P(Yes),P(No),Decision
0,Weak,High,99.99%,0.01%,Hold (rain)
1,Strong,High,99.03%,0.97%,Hold (rain)
2,Weak,Low,49.75%,50.25%,Deliver
3,Strong,Low,1.47%,98.53%,Deliver


In [ ]:
def penalized_nll(v, lam=1.0):
    p = sigmoid(b + X @ np.array([w1, w2]))
    eps = 1e-12
    nll = -np.sum(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    return nll + 0.5 * lam * (w1 ** 2 + w2 ** 2)

res = minimize(penalized_nll, np.zeros(3), method="BFGS")
w_reg = res.x
print("L2-regularized weights [bias, w_wind, w_humidity]:", np.round(w_reg, 4))

reg_rows = []
for wind, hum in cases:
    p = lr_prob(wind, hum, w_reg)
    reg_rows.append({"Wind": wind, "Humidity": hum, "P(Yes)": f"{p:.2%}",
                     "Decision": "Hold (rain)" if p >= 0.5 else "Deliver"})
pd.DataFrame(reg_rows)

L2-regularized weights [bias, w_wind, w_humidity]: [ 0.235  -0.1792  0.6309]


,Wind,Humidity,P(Yes),Decision
0,Weak,High,70.39%,Hold (rain)
1,Strong,High,66.52%,Hold (rain)
2,Weak,Low,55.85%,Hold (rain)
3,Strong,Low,51.39%,Hold (rain)


## Part 4: 

In [14]:
def nb_pred(wind, hum):
    scores = {c: prior[c] * laplace_likelihood("Wind", wind, c) * laplace_likelihood("Humidity", hum, c)
              for c in classes}
    return max(scores, key=scores.get)

def tree_pred(wind, hum):
    return tree_predict(wind, hum)

def lr_pred(wind, hum):
    return "Yes" if lr_prob(wind, hum, w_final) >= 0.5 else "No"

comparison = pd.DataFrame([
    {"Wind": wd, "Humidity": hm,
     "Naive Bayes": nb_pred(wd, hm),
     "Decision tree": tree_pred(wd, hm),
     "Logistic regression": lr_pred(wd, hm)}
    for wd, hm in cases
])
comparison

,Wind,Humidity,Naive Bayes,Decision tree,Logistic regression
0,Weak,High,Yes,Yes,Yes
1,Strong,High,Yes,Yes,Yes
2,Weak,Low,No,No,No
3,Strong,Low,No,No,No


In [15]:
for name, fn in [("Naive Bayes", nb_pred), ("Decision tree", tree_pred), ("Logistic regression", lr_pred)]:
    preds = [fn(wd, hm) for wd, hm in zip(data["Wind"], data["Humidity"])]
    acc = np.mean([p == t for p, t in zip(preds, data["Rain"])])
    print(f"{name:<20} training accuracy = {acc:.0%}")

Naive Bayes          training accuracy = 80%
Decision tree        training accuracy = 80%
Logistic regression  training accuracy = 80%
